# Baseline 3: prompted intent parser (GPU step)

Each parser reads every task's request + dataset schema and writes an intent contract (vocabulary v1.0).
One run parses with three models in turn, with the **same prompt and decoding**:
- `Qwen2.5-7B-Instruct` (4-bit): **Baseline 3** (`intent_prompted_7b`)
- `Qwen2.5-3B-Instruct` and `Qwen2.5-1.5B-Instruct` (fp16): size ablation (`intent_prompted_3b`, `intent_prompted_1_5b`), the prompted counterparts of the fine-tuned parsers
Only **parsing** runs here. Validation, contract metrics and the verifier run on the laptop.

**Settings:** Accelerator **GPU T4 x1**, Internet **on**. Private repo: Kaggle secret `GITHUB_TOKEN`.
Then *Save Version → Save & Run All* (roughly 1–2 h for all three). Download the three `parse_<model>.zip` files from the Output tab.

Prompt `p1-v1`: few-shot examples are 5 **dev** tasks (one per transformation type); greedy decoding, no constrained decoding (README §12).

In [1]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
# (model, load in 4-bit, system name used on the laptop). Remove lines to run fewer parsers.
PARSERS = [
    ("Qwen/Qwen2.5-7B-Instruct", True, "intent_prompted_7b"),      # Baseline 3; 7B fp16 doesn't fit a 16 GB T4
    ("Qwen/Qwen2.5-3B-Instruct", False, "intent_prompted_3b"),     # size ablation
    ("Qwen/Qwen2.5-1.5B-Instruct", False, "intent_prompted_1_5b"), # size ablation
]
BATCH = 4
LIMIT = None              # e.g. 4 for a quick dry run
def out_dir(model_id):
    return "/kaggle/working/parse_" + model_id.split("/")[-1].lower().replace(".", "")
for m, q, s in PARSERS:
    print(f"{s:<22} {m:<28} 4-bit={q}  ->  {out_dir(m)}")

intent_prompted_7b     Qwen/Qwen2.5-7B-Instruct     4-bit=True  ->  /kaggle/working/parse_qwen25-7b-instruct
intent_prompted_3b     Qwen/Qwen2.5-3B-Instruct     4-bit=False  ->  /kaggle/working/parse_qwen25-3b-instruct
intent_prompted_1_5b   Qwen/Qwen2.5-1.5B-Instruct   4-bit=False  ->  /kaggle/working/parse_qwen25-15b-instruct


In [2]:
# ---------------- Get the code ----------------
import os, shutil, subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s (%cd)"], capture_output=True, text=True).stdout)
NEEDED = ["src/igt/agents/intent_parser.py", "src/igt/contracts/schema.py", "eval/parse_contracts.py",
          "eval/generate.py", "eval/prompts/schema_b1-v1.json", "eval/tasks"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"Not in the GitHub repo (push them first): {missing}"
print("all needed files present")

no GITHUB_TOKEN secret; cloning without a token (public repo)


Cloning into '/kaggle/working/repo'...


c227e04a994df24017e92e672b4cc364049e2093 baseline 3 - ablation (Fri Oct 30 00:10:09 2026 +0530)

all needed files present


In [3]:
# ---------------- Libraries: bitsandbytes for 4-bit (pydantic 2 is needed for validation) ----------------
if any(q for _, q, _ in PARSERS):
    subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
import sys, torch, pydantic
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator")
print("pydantic", pydantic.VERSION)
assert pydantic.VERSION.startswith("2"), "pydantic 2 required"
sys.path.insert(0, "src")
from igt.agents.intent_parser import PROMPT_HASH, PARSER_PROMPT_VERSION
print("parser prompt:", PARSER_PROMPT_VERSION, PROMPT_HASH)

GPU: Tesla T4
pydantic 2.13.5
parser prompt: p1-v1 1b8d8783f898


In [4]:
# ---------------- Parse every task with each parser (resumable; one subprocess per model frees the GPU) ----------------
import time
for model_id, q, system in PARSERS:
    t0 = time.time()
    cmd = ["python", "eval/parse_contracts.py", "--model", model_id, "--out", out_dir(model_id), "--batch", str(BATCH)]
    if q:
        cmd.append("--load-in-4bit")
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    print(f"===== {system}: {model_id}")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-1500:])
    print(p.stderr[-1500:])
    assert p.returncode == 0, system
    print(f"{system} done in {(time.time() - t0) / 60:.1f} min")

===== intent_prompted_7b: Qwen/Qwen2.5-7B-Instruct
rsed 52/120 (690s) statuses: ['ok', 'invalid_contract', 'ok', 'ok']
parsed 56/120 (740s) statuses: ['ok', 'invalid_contract', 'ok', 'ok']
parsed 60/120 (791s) statuses: ['ok', 'invalid_contract', 'ok', 'ok']
parsed 64/120 (842s) statuses: ['ok', 'invalid_contract', 'invalid_json', 'invalid_contract']
parsed 68/120 (895s) statuses: ['invalid_contract', 'ok', 'invalid_contract', 'invalid_contract']
parsed 72/120 (939s) statuses: ['invalid_contract', 'ok', 'invalid_contract', 'invalid_contract']
parsed 76/120 (987s) statuses: ['ok', 'ok', 'ok', 'ok']
parsed 80/120 (1036s) statuses: ['ok', 'ok', 'ok', 'ok']
parsed 84/120 (1095s) statuses: ['ok', 'ok', 'invalid_json', 'ok']
parsed 88/120 (1149s) statuses: ['invalid_contract', 'ok', 'ok', 'ok']
parsed 92/120 (1202s) statuses: ['invalid_contract', 'ok', 'ok', 'ok']
parsed 96/120 (1279s) statuses: ['invalid_contract', 'invalid_contract', 'invalid_contract', 'ok']
parsed 100/120 (1335s) statuse

In [5]:
# ---------------- Quick look and zip ----------------
import json, collections
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout
for model_id, q, system in PARSERS:
    OUT = out_dir(model_id)
    R = [json.loads(l) for l in open(f"{OUT}/parses.jsonl")]
    print(f"{system}: {len(R)} parses | status {dict(collections.Counter(r['parse_status'] for r in R))} | truncated {sum(r['truncated'] for r in R)}")
    open(f"{OUT}/git_commit.txt", "w").write(commit)
    open(f"{OUT}/system.txt", "w").write(system)
    print("   saved:", shutil.make_archive(OUT, "zip", OUT))
print()
print(R[0]["task_id"], "->"); print(R[0]["raw_output"][:1200])
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)

intent_prompted_7b: 120 parses | status {'ok': 79, 'invalid_contract': 36, 'invalid_json': 5} | truncated 0
   saved: /kaggle/working/parse_qwen25-7b-instruct.zip
intent_prompted_3b: 120 parses | status {'ok': 67, 'invalid_contract': 50, 'invalid_json': 3} | truncated 0
   saved: /kaggle/working/parse_qwen25-3b-instruct.zip
intent_prompted_1_5b: 120 parses | status {'invalid_contract': 62, 'ok': 55, 'invalid_json': 3} | truncated 1
   saved: /kaggle/working/parse_qwen25-15b-instruct.zip

adult_col_001 ->
```json
{"contract_version": "1.0",
 "must_change": [
  {"type": "column_value_relation", "target": "net_capital", "fn": "subtract", "args": [{"col": "capital-gain"}, {"col": "capital-loss"}]}
 ],
 "must_preserve": [
  {"type": "columns_unchanged", "target": "*", "except": ["capital-gain", "capital-loss"]},
  {"type": "row_count", "target": null, "op": "unchanged"}
 ]
}
```
